In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from datetime import datetime
import joblib

# Step 1: Load Dataset
# Assuming `df` is your dataset with QoS and user information.
# df = pd.read_csv('your_dataset.csv') # Uncomment if loading from a file

# Sample mock data structure if dataset isn't loaded (replace this with your actual data if needed)
# df = pd.DataFrame(...)

# Step 2: Exploratory Data Analysis (EDA)
print("Exploratory Data Analysis:\n")

# Summary statistics for numerical features
print("\nSummary Statistics for Numerical Columns:\n", df.describe().T)

# Frequency counts for categorical columns
categorical_cols = ['Service_Group', 'Service_Name', 'Device_Type']
for col in categorical_cols:
    print(f"\nFrequency count for {col}:\n", df[col].value_counts())

# Distribution Analysis Visualizations

# Count of Device Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Device_Type', palette='viridis')
plt.title('Distribution of Device Types')
plt.xlabel('Device Type')
plt.ylabel('Count')
plt.show()

# Count of Application Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Service_Name', palette='plasma')
plt.title('Distribution of Application Types')
plt.xlabel('Application Type')
plt.ylabel('Count')
plt.show()

# Signal Strength Distribution per Application Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Service_Name', y='Signal_Strength', palette='coolwarm')
plt.title('Signal Strength Distribution per Application Type')
plt.xlabel('Application Type')
plt.ylabel('Signal Strength (dBm)')
plt.show()

# Latency Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Latency'], bins=30, kde=True, color='skyblue')
plt.title('Latency Distribution')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.show()

# Bandwidth Usage Distribution per Service Group
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Service_Group', y='Bandwidth_Speed_per_sec_Mbps', palette='muted')
plt.title('Bandwidth Usage per Service Group')
plt.xlabel('Service Group')
plt.ylabel('Bandwidth Speed (Mbps)')
plt.show()

# Packet Loss Rate vs Priority Score
plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x='Packet_Loss_Rate', y='Priority_Score', hue='Service_Group', palette='cool')
plt.title('Packet Loss Rate vs Priority Score')
plt.xlabel('Packet Loss Rate')
plt.ylabel('Priority Score')
plt.legend(title='Service Group')
plt.show()

# Buffer Occupancy Distribution by Time of Day
df['hour'] = df['Timestamp'].dt.hour  # Extract hour from timestamp
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='hour', y='Buffer_Occupancy', palette='Set2')
plt.title('Buffer Occupancy by Time of Day')
plt.xlabel('Hour of Day')
plt.ylabel('Buffer Occupancy')
plt.show()

# Correlation Heatmap for Numerical Features
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of QoS Features')
plt.show()

# Step 3: Statistical Analysis
print("Statistical Analysis:\n")
print("Correlation matrix:\n", df.corr())

# Step 4: Data Preprocessing

# Extract day_of_week from Timestamp (if not already extracted)
df['day_of_week'] = df['Timestamp'].dt.day_name()

# Check for missing values
print("\nMissing values before handling:\n", df.isnull().sum())

# Handle missing values
# Fill numerical columns with median and categorical columns with mode
numerical_cols = ['Signal_Strength', 'Packet_Loss_Rate', 'Latency', 'Jitter_ms',
                  'Bandwidth_Speed_per_sec_Mbps', 'Buffer_Occupancy', 'Usage_Minutes', 'Usage_Percentage']
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols + ['day_of_week']:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates (excluding Hashid for uniqueness)
df = df.drop_duplicates(subset=[col for col in df.columns if col != 'Hashid'])

print("\nMissing values after handling:\n", df.isnull().sum())
print("Number of rows after removing duplicates:", df.shape[0])

# Label Encoding for categorical features
label_encoders = {}
for col in categorical_cols + ['day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Standardize numerical columns
scaler = StandardScaler()
df[numerical_cols + ['hour']] = scaler.fit_transform(df[numerical_cols + ['hour']])

# Define features (X) and target (y) with Hashid for tracking
X = df.drop(columns=['Hashid', 'Timestamp', 'Priority_Score'])  # Drop unnecessary columns and target column
y = np.random.choice(['Low', 'Medium', 'High'], size=df.shape[0])  # Mock Priority_Score for ordinal target

# Train-test split with Hashid tracking
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Separate Hashid for tracking
train_hashid = X_train['Hashid']
test_hashid = X_test['Hashid']

# Drop Hashid from the feature set before model training
X_train = X_train.drop(columns=['Hashid'])
X_test = X_test.drop(columns=['Hashid'])

# Step 5: Ordinal Classification Model with Random Forest and Hyperparameter Tuning

# Define the parameter grid for optimization
param_grid = {
    'n_estimators': [50, 100, 150],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# Initialize the Random Forest with GridSearch for hyperparameter tuning
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Retrieve the best model
best_rf = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Step 6: Evaluation on Test Set
y_pred = best_rf.predict(X_test)

# Attach Hashid to the predictions to map results back to each user/device
predictions_with_hashid = pd.DataFrame({
    'Hashid': test_hashid,
    'Actual_Priority': y_test,
    'Predicted_Priority': y_pred
})

# Display prediction results with Hashid for tracking
print("\nPredictions with Hashid Tracking:\n", predictions_with_hashid.head())

# Classification Report and Confusion Matrix
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Step 7: Feature Importance Visualization
importances = best_rf.feature_importances_
features = X_train.columns
indices = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.title('Feature Importance')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

# Step 8: Save the Model and Encoders if Needed
# Save model
joblib.dump(best_rf, 'best_rf_model.joblib')
# Save label encoders and scaler for future use
for col, encoder in label_encoders.items():
    joblib.dump(encoder, f'label_encoder_{col}.joblib')
joblib.dump(scaler, 'scaler.joblib')


In [ ]:
# Packet Loss Rate by Service Group
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Service_Group', y='Packet_Loss_Rate', palette='cool')
plt.title('Packet Loss Rate by Service Group')
plt.xlabel('Service Group')
plt.ylabel('Packet Loss Rate')
plt.show()

# Buffer Occupancy Distribution by Day of the Week
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='day_of_week', y='Buffer_Occupancy', palette='Set2')
plt.title('Buffer Occupancy by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Buffer Occupancy')
plt.show()

# Buffer Occupancy Distribution by Hour of the Day
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='hour', y='Buffer_Occupancy', palette='Set3')
plt.title('Buffer Occupancy by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('Buffer Occupancy')
plt.show()


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from datetime import datetime

# Step 1: Load Dataset
# Assuming `df` is your dataset with QoS and user information.
# df = pd.read_csv('your_dataset.csv') # Uncomment if loading from a file

# Sample mock data structure if dataset isn't loaded (replace this with your actual data if needed)
# df = pd.DataFrame(...)

# Step 2: Exploratory Data Analysis (EDA)
print("Exploratory Data Analysis:\n")

# Summary statistics for numerical features
print("\nSummary Statistics for Numerical Columns:\n", df.describe().T)

# Frequency counts for categorical columns
categorical_cols = ['Service_Group', 'Service_Name', 'Device_Type']
for col in categorical_cols:
    print(f"\nFrequency count for {col}:\n", df[col].value_counts())

# Distribution Analysis Visualizations

# Count of Device Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Device_Type', palette='viridis')
plt.title('Distribution of Device Types')
plt.xlabel('Device Type')
plt.ylabel('Count')
plt.show()

# Count of Application Types
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Service_Name', palette='plasma')
plt.title('Distribution of Application Types')
plt.xlabel('Application Type')
plt.ylabel('Count')
plt.show()

# Signal Strength Distribution per Application Type
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Service_Name', y='Signal_Strength', palette='coolwarm')
plt.title('Signal Strength Distribution per Application Type')
plt.xlabel('Application Type')
plt.ylabel('Signal Strength (dBm)')
plt.show()

# Latency Distribution
plt.figure(figsize=(8, 5))
sns.histplot(df['Latency'], bins=30, kde=True, color='skyblue')
plt.title('Latency Distribution')
plt.xlabel('Latency (ms)')
plt.ylabel('Frequency')
plt.show()

# Bandwidth Usage Distribution per Service Group
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='Service_Group', y='Bandwidth_Speed_per_sec_Mbps', palette='muted')
plt.title('Bandwidth Usage per Service Group')
plt.xlabel('Service Group')
plt.ylabel('Bandwidth Speed (Mbps)')
plt.show()

# Packet Loss Rate by Service Group
plt.figure(figsize=(8, 6))
sns.boxplot(data=df, x='Service_Group', y='Packet_Loss_Rate', palette='cool')
plt.title('Packet Loss Rate by Service Group')
plt.xlabel('Service Group')
plt.ylabel('Packet Loss Rate')
plt.show()

# Buffer Occupancy Distribution by Day of the Week
df['day_of_week'] = df['Timestamp'].dt.day_name()
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='day_of_week', y='Buffer_Occupancy', palette='Set2')
plt.title('Buffer Occupancy by Day of the Week')
plt.xlabel('Day of the Week')
plt.ylabel('Buffer Occupancy')
plt.show()

# Buffer Occupancy Distribution by Hour of the Day
df['hour'] = df['Timestamp'].dt.hour
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='hour', y='Buffer_Occupancy', palette='Set3')
plt.title('Buffer Occupancy by Hour of the Day')
plt.xlabel('Hour of the Day')
plt.ylabel('Buffer Occupancy')
plt.show()

# Correlation Heatmap for Numerical Features
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of QoS Features')
plt.show()

# Step 3: Statistical Analysis
print("Statistical Analysis:\n")
print("Correlation matrix:\n", df.corr())

# Step 4: Data Preprocessing

# Check for missing values
print("\nMissing values before handling:\n", df.isnull().sum())

# Handle missing values
# Fill numerical columns with median and categorical columns with mode
numerical_cols = ['Signal_Strength', 'Packet_Loss_Rate', 'Latency', 'Jitter_ms',
                  'Bandwidth_Speed_per_sec_Mbps', 'Buffer_Occupancy', 'Usage_Minutes', 'Usage_Percentage']
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols + ['day_of_week']:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates (excluding Hashid for uniqueness)
df = df.drop_duplicates(subset=[col for col in df.columns if col != 'Hashid'])

print("\nMissing values after handling:\n", df.isnull().sum())
print("Number of rows after removing duplicates:", df.shape[0])

# Label Encoding for categorical features
label_encoders = {}
for col in categorical_cols + ['day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Standardize numerical columns
scaler = StandardScaler()
df[numerical_cols + ['hour']] = scaler.fit_transform(df[numerical_cols + ['hour']])

# Saving encoders and scaler for potential future use
import joblib
for col, encoder in label_encoders.items():
    joblib.dump(encoder, f'label_encoder_{col}.joblib')
joblib.dump(scaler, 'scaler.joblib')


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
import seaborn as sns

# Sample data loading (replace with actual dataset)
df = pd.DataFrame({
    'device_type': ['Laptop', 'Mobile', 'Tablet', 'PC'],
    'usage_minutes': [120, 300, 150, 200],
    'buffer_occupancy': [0.6, 0.4, 0.3, 0.5],
    'signal_strength': [-50, -70, -45, -60],
    'latency': [30, 50, 20, 40],
    'jitter_ms': [5, 10, 3, 7],
    'bandwidth_speed_per_sec_mbps': [10, 5, 8, 15],
    'packet_loss_rate': [0.02, 0.04, 0.01, 0.03],
    'timestamp': pd.to_datetime(['2023-01-01 08:00', '2023-01-01 12:00', '2023-01-01 18:00', '2023-01-01 22:00'])
})

# Step 1: Feature Engineering
# Extract day of the week and hour from timestamp for pattern analysis
df['day_of_week'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour

# Label encode categorical features
label_encoders = {}
for col in ['device_type', 'day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Standardize numerical columns
numerical_cols = ['usage_minutes', 'buffer_occupancy', 'signal_strength', 'latency', 'jitter_ms', 'bandwidth_speed_per_sec_mbps', 'packet_loss_rate', 'hour']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Step 2: Clustering to Identify User Segments
# Determine optimal number of clusters using the elbow method
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, random_state=42)
    kmeans.fit(df[numerical_cols])
    wcss.append(kmeans.inertia_)

# Plot the elbow curve
plt.figure(figsize=(8, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--')
plt.title('Elbow Method for Optimal k')
plt.xlabel('Number of Clusters')
plt.ylabel('WCSS')
plt.show()

# Fit KMeans with optimal number of clusters
optimal_k = 3  # Based on elbow plot
kmeans = KMeans(n_clusters=optimal_k, random_state=42)
df['user_cluster'] = kmeans.fit_predict(df[numerical_cols])

# Step 3: Recommender Logic Based on Clusters and Device Type
# Define a priority score based on cluster and device_type
def recommend_bandwidth(row):
    if row['user_cluster'] == 0:
        return 'High' if row['device_type'] == 0 else 'Medium'
    elif row['user_cluster'] == 1:
        return 'Medium' if row['device_type'] == 1 else 'Low'
    else:
        return 'High' if row['device_type'] in [0, 2] else 'Medium'

df['bandwidth_priority'] = df.apply(recommend_bandwidth, axis=1)

# Display recommendations
print("Recommended Bandwidth Priority per User Segment:")
print(df[['device_type', 'user_cluster', 'bandwidth_priority']])


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Step 1: Load Dataset
df = pd.read_csv('qos.csv')

# Ensure timestamp is parsed as datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Feature Engineering - Extract day and hour from timestamp
df['day_of_week'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour

# Label encode categorical features
label_encoders = {}
for col in ['service_group', 'service_name', 'device_type', 'day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Define Priority based on Heuristic Rules
def generate_priority(row):
    if row['service_group'] == label_encoders['service_group'].transform(['software'])[0]:
        return 'High'
    elif row['service_group'] in label_encoders['service_group'].transform(['streaming', 'gaming']):
        return 'Medium'
    else:
        return 'Low'

df['priority'] = df.apply(generate_priority, axis=1)

# Map priority labels to ordinal values
priority_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['priority'] = df['priority'].map(priority_mapping)

# Preprocessing - Standardize numerical features
numerical_cols = ['usage_minutes', 'buffer_occupancy', 'signal_strength', 'latency', 'jitter_ms', 'bandwidth_speed_per_sec_mbps', 'packet_loss_rate', 'hour']
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Train-test Split
X = df.drop(columns=['timestamp', 'priority'])
y = df['priority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Ordinal Classification with Random Forest
# Define the parameter grid for Random Forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Initialize Random Forest Classifier
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Evaluation on Test Set
y_pred = best_rf.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Feature Importance Visualization
importances = best_rf.feature_importances_
features = X_train.columns
indices = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.title('Feature Importance')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

# Save Model and Encoders
joblib.dump(best_rf, 'best_rf_model.joblib')
for col, encoder in label_encoders.items():
    joblib.dump(encoder, f'label_encoder_{col}.joblib')
joblib.dump(scaler, 'scaler.joblib')


In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import joblib

# Step 1: Load Dataset
df = pd.read_csv('qos.csv')

# Step 2: Exploratory Data Analysis (EDA)
# Check basic info
print("Dataset Info:")
df.info()

# Summary statistics for numerical features
print("\nSummary Statistics for Numerical Columns:\n", df.describe().T)

# Check unique values in categorical features
categorical_cols = ['service_group', 'service_name', 'device_type']
for col in categorical_cols:
    print(f"\nUnique values in {col}:\n", df[col].unique())

# Distribution of Service Group
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='service_group', palette='viridis')
plt.title('Distribution of Service Group')
plt.xlabel('Service Group')
plt.ylabel('Count')
plt.show()

# Distribution Analysis of Network-Related Features
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='service_group', y='signal_strength', palette='coolwarm')
plt.title('Signal Strength Distribution per Service Group')
plt.xlabel('Service Group')
plt.ylabel('Signal Strength (dBm)')
plt.show()

# Correlation Heatmap for Numerical Features
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='viridis', fmt='.2f')
plt.title('Correlation Heatmap of QoS Features')
plt.show()

# Step 3: Data Preprocessing
# Ensure timestamp is parsed as datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])

# Extract day and hour from timestamp
df['day_of_week'] = df['timestamp'].dt.day_name()
df['hour'] = df['timestamp'].dt.hour

# Check for missing values
print("\nMissing values before handling:\n", df.isnull().sum())

# Handle missing values
# Fill numerical columns with median and categorical columns with mode
numerical_cols = ['usage_minutes', 'buffer_occupancy', 'signal_strength', 'latency', 'jitter_ms',
                  'bandwidth_speed_per_sec_mbps', 'packet_loss_rate', 'hour']
for col in numerical_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in categorical_cols + ['day_of_week']:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Remove duplicates
df = df.drop_duplicates()

print("\nMissing values after handling:\n", df.isnull().sum())

# Label encode categorical features
label_encoders = {}
for col in ['service_group', 'service_name', 'device_type', 'day_of_week']:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

# Standardize numerical features
scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])

# Step 4: Define Priority based on Heuristic Rules
# Map service groups to priority levels
def generate_priority(row):
    if row['service_group'] == label_encoders['service_group'].transform(['software'])[0]:
        return 'High'
    elif row['service_group'] in label_encoders['service_group'].transform(['streaming', 'gaming']):
        return 'Medium'
    else:
        return 'Low'

df['priority'] = df.apply(generate_priority, axis=1)

# Map priority labels to ordinal values
priority_mapping = {'Low': 0, 'Medium': 1, 'High': 2}
df['priority'] = df['priority'].map(priority_mapping)

# Step 5: Train-test Split
X = df.drop(columns=['timestamp', 'priority'])
y = df['priority']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 6: Ordinal Classification with Random Forest
# Define the parameter grid for Random Forest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Initialize Random Forest Classifier
rf = RandomForestClassifier(random_state=42, class_weight='balanced')
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=2)
grid_search.fit(X_train, y_train)

# Best model
best_rf = grid_search.best_estimator_
print("\nBest Parameters:", grid_search.best_params_)

# Step 7: Evaluation on Test Set
y_pred = best_rf.predict(X_test)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['Low', 'Medium', 'High']))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Feature Importance Visualization
importances = best_rf.feature_importances_
features = X_train.columns
indices = np.argsort(importances)

plt.figure(figsize=(10, 8))
plt.title('Feature Importance')
plt.barh(range(len(indices)), importances[indices], align='center')
plt.yticks(range(len(indices)), [features[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()

# Step 8: Save Model and Encoders
joblib.dump(best_rf, 'best_rf_model.joblib')
for col, encoder in label_encoders.items():
    joblib.dump(encoder, f'label_encoder_{col}.joblib')
joblib.dump(scaler, 'scaler.joblib')
